# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qurban-12/ML_internship_Qurban/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked content-review queue

The model score is used to prioritize content for human review, not to automatically change content.

The highest-scored items are reviewed first because the client-grouped holdout showed directional ranking signal for the observed declining-label task (ROC-AUC 0.6157; P@20 0.70; P@50 0.72; P@100 0.70).

The main action is `review_for_refresh`.

Reason codes make the queue easier for a human reviewer to understand:
- `high_model_score`: high priority according to the validated model ranking.
- `stale_content`: content has a relatively high number of days since its last update.
- `low_ctr`: observed click-through rate is low.
- `low_position`: average search position is relatively weak.
- `older_content`: content has been live for a long time.

A reason code is a review signal, not proof that refreshing the content will improve performance.
The reason codes use relative thresholds within the evaluated queue, so they indicate review signals rather than universal business cutoffs.

In [21]:
from pathlib import Path
import subprocess

repo_path = Path("/content/ML_internship_Qurban")

if not repo_path.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/qurban-12/ML_internship_Qurban.git",
            str(repo_path),
        ],
        check=True,
    )

print("Repository:", repo_path)
print(
    "Preparation script exists:",
    (repo_path / "scripts/01_prepare_features.py").exists()
)

Repository: /content/ML_internship_Qurban
Preparation script exists: True


In [22]:
# W07 — Ranked actions + reason codes
# Locate the repository, recreate the Week-5 prepared dataset,
# and reproduce the Week-6 client-grouped model.

import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression



# 1. Locate the repository

possible_repo_paths = [
    Path("/content/ML_internship_Qurban"),
    Path("/content/ML_internship_Qurban-main"),
    Path.cwd() / "ML_internship_Qurban",
    Path.cwd(),
]

repo_path = None

for candidate in possible_repo_paths:
    if (
        (candidate / "scripts" / "01_prepare_features.py").exists()
        and (candidate / "work").exists()
    ):
        repo_path = candidate
        break

if repo_path is None:
    raise FileNotFoundError(
        "Could not locate the ML_internship_Qurban repository. "
        "Make sure the GitHub repository is cloned/opened in this Colab session."
    )

print("Repository found:", repo_path.resolve())



# 2. Recreate the exact Week-5 prepared dataset


result = subprocess.run(
    ["python", "scripts/01_prepare_features.py"],
    cwd=repo_path,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("STDERR:")
    print(result.stderr)

assert result.returncode == 0, "01_prepare_features.py failed."

PREPARED_PATH = repo_path / "data/processed/refresh_feature_vector.csv"

print("Prepared file exists:", PREPARED_PATH.exists())

assert PREPARED_PATH.exists(), (
    f"Prepared dataset not found: {PREPARED_PATH}"
)

df = pd.read_csv(PREPARED_PATH)

print("\nPrepared dataset shape:", df.shape)
print("Target exists:", "is_declining_label" in df.columns)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(round(df["is_declining_label"].mean() * 100, 2), "%")



# 3. Same Week-6 feature set


numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

feature_columns = numeric_features + categorical_features

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

assert not missing_features, (
    f"Missing Week-6 features: {missing_features}"
)

X = df[feature_columns].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()



# 4. Recreate the honest client-grouped split


gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

test_grouped = df.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("\nGrouped split")
print("----------------")
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Shared clients:", len(train_clients & test_clients))
print(
    "Test decline rate:",
    round(y_test_grouped.mean() * 100, 2),
    "%"
)



# 5. Same preprocessing + Logistic Regression


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

grouped_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])



# 6. Train and score


grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

test_grouped["model_score"] = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

test_grouped = test_grouped.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

test_grouped["rank"] = np.arange(
    1,
    len(test_grouped) + 1
)



# 7. Reason codes


score_75 = test_grouped["model_score"].quantile(0.75)
update_median = test_grouped["days_since_last_update"].median()
ctr_median = test_grouped["ctr"].median()
position_median = test_grouped["avg_position"].median()
age_median = test_grouped["content_age_days"].median()

reason_codes = []

for _, row in test_grouped.iterrows():

    reasons = []

    if row["model_score"] >= score_75:
        reasons.append("high_model_score")

    if row["days_since_last_update"] >= update_median:
        reasons.append("stale_content")

    if row["ctr"] <= ctr_median:
        reasons.append("low_ctr")

    if row["avg_position"] >= position_median:
        reasons.append("low_position")

    if row["content_age_days"] >= age_median:
        reasons.append("older_content")

    if not reasons:
        reasons.append("model_priority_only")

    reason_codes.append(";".join(reasons))

test_grouped["reason_code"] = reason_codes



# 8. Human-review action


test_grouped["action"] = "review_for_refresh"

test_grouped["human_review_required"] = True

test_grouped["confidence_note"] = (
    "Directional ranking signal only; "
    "human review required before any action."
)



# 9. Final queue


ranked_queue = test_grouped[
    [
        "rank",
        "content_id",
        "client_id",
        "model_score",
        "action",
        "reason_code",
        "confidence_note",
        "human_review_required",
    ]
].copy()


print("\nRanked queue size:", len(ranked_queue))

print("\nTop 20 review priorities:")
display(ranked_queue.head(20))

Repository found: /content/ML_internship_Qurban
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/ML_internship_Qurban/data/processed/refresh_feature_vector.csv

Prepared file exists: True

Prepared dataset shape: (30000, 52)
Target exists: True

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
54.21 %

Grouped split
----------------
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Shared clients: 0
Test decline rate: 51.1 %

Ranked queue size: 6163

Top 20 review priorities:


,rank,content_id,client_id,model_score,action,reason_code,confidence_note,human_review_required
0,1,content_a928cb66d230,client_f369cb89fc,0.957424,review_for_refresh,high_model_score;stale_content;low_ctr,Directional ranking signal only; human review ...,True
1,2,content_7be5f150dc65,client_f369cb89fc,0.954100,review_for_refresh,high_model_score;stale_content;low_ctr,Directional ranking signal only; human review ...,True
2,3,content_c82bc0c24241,client_f369cb89fc,0.947583,review_for_refresh,high_model_score;low_ctr,Directional ranking signal only; human review ...,True
3,4,content_87c007fb5c26,client_f369cb89fc,0.944469,review_for_refresh,high_model_score;stale_content;low_ctr,Directional ranking signal only; human review ...,True
4,5,content_41baf0722ad9,client_8527a891e2,0.942681,review_for_refresh,high_model_score;stale_content;low_ctr;low_pos...,Directional ranking signal only; human review ...,True
5,6,content_5d77d3077984,client_f369cb89fc,0.940838,review_for_refresh,high_model_score;stale_content;low_ctr,Directional ranking signal only; human review ...,True
6,7,content_b5e9e6453511,client_f369cb89fc,0.935936,review_for_refresh,high_model_score;stale_content;low_ctr,Directional ranking signal only; human review ...,True
7,8,content_8ba781dafa55,client_8527a891e2,0.935417,review_for_refresh,high_model_score;stale_content;low_ctr;older_c...,Directional ranking signal only; human review ...,True
8,9,content_e988c1699454,client_8527a891e2,0.935177,review_for_refresh,high_model_score;stale_content;low_ctr;low_pos...,Directional ranking signal only; human review ...,True
9,10,content_11a4f985f14d,client_f369cb89fc,0.932941,review_for_refresh,high_model_score;stale_content;low_ctr,Directional ranking signal only; human review ...,True


### Archetype → action mapping

The action mapping is intentionally conservative. The model does not automatically assign a content change; it assigns a review priority.

| Review archetype | Signals | Suggested human review |
|---|---|---|
| High-priority refresh candidate | High model score plus one or more supporting reason codes | Review freshness, search intent, content quality, and recent performance |
| Stale content | `stale_content` reason code | Check whether information, examples, or recommendations are outdated |
| Low CTR | `low_ctr` reason code | Check title, search intent alignment, snippet relevance, and SERP context |
| Weak position signal | `low_position` reason code | Check search intent, topical coverage, internal linking, and competing results |
| Older content | `older_content` reason code | Check whether the page needs factual or structural updates |
| Model-only priority | `model_priority_only` | Review the page manually before deciding whether any action is justified |

These mappings are review guidelines, not automatic optimization rules. Multiple signals increase the reason to inspect an item, but they do not prove that a specific change will improve performance.

In [23]:
# Archetype → action mapping used for human review.

archetype_actions = pd.DataFrame([
    {
        "archetype": "High-priority refresh candidate",
        "signals": "high_model_score + supporting reason code",
        "suggested_review": "Review freshness, intent, quality, and recent performance",
        "automation_allowed": False
    },
    {
        "archetype": "Stale content",
        "signals": "stale_content",
        "suggested_review": "Check whether information or recommendations are outdated",
        "automation_allowed": False
    },
    {
        "archetype": "Low CTR",
        "signals": "low_ctr",
        "suggested_review": "Check title, intent alignment, snippet relevance, and SERP context",
        "automation_allowed": False
    },
    {
        "archetype": "Weak position signal",
        "signals": "low_position",
        "suggested_review": "Check intent, topical coverage, internal links, and competition",
        "automation_allowed": False
    },
    {
        "archetype": "Older content",
        "signals": "older_content",
        "suggested_review": "Check whether the page needs factual or structural updates",
        "automation_allowed": False
    },
    {
        "archetype": "Model-only priority",
        "signals": "model_priority_only",
        "suggested_review": "Review manually before deciding whether action is justified",
        "automation_allowed": False
    }
])

display(archetype_actions)
print("All mapped actions require human review:",
      (~archetype_actions["automation_allowed"]).all())

,archetype,signals,suggested_review,automation_allowed
0,High-priority refresh candidate,high_model_score + supporting reason code,"Review freshness, intent, quality, and recent ...",False
1,Stale content,stale_content,Check whether information or recommendations a...,False
2,Low CTR,low_ctr,"Check title, intent alignment, snippet relevan...",False
3,Weak position signal,low_position,"Check intent, topical coverage, internal links...",False
4,Older content,older_content,Check whether the page needs factual or struct...,False
5,Model-only priority,model_priority_only,Review manually before deciding whether action...,False


All mapped actions require human review: True


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended to help SEO and content teams prioritize which content items should receive human review first.

The model score is used as a ranking signal for the observed declining-label task. On the client-grouped holdout containing 7 unseen clients, the model showed ROC-AUC of 0.6157, with precision of 0.70 at the top 20 and 0.72 at the top 50.

The queue should therefore be used for decision-support and review prioritization, not as an automatic prediction of future traffic decline.

### Limits

The target label is based on observed changes in impressions between two recent 30-day periods. Several model inputs are 90-day aggregates, so this evaluation does not establish reliable future prediction.

The client-grouped holdout is more realistic than a random row split because it contains unseen clients. Performance was lower on the grouped split than on the random split, showing that client overlap can make results look more optimistic.

The model produces both false positives and false negatives. A high score does not guarantee that content is actually declining, and a low score does not guarantee that content is healthy.

The reason codes are review signals, not proof that a particular content change will improve performance.

The playbook does not establish that refreshing content causes improvements in traffic, rankings, CTR, or engagement.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 — Recalculate the validated evidence used to define intended use.

from sklearn.metrics import roc_auc_score
import numpy as np

# Generate predictions on the same client-grouped holdout used in Week 6.
grouped_test_scores = grouped_model.predict_proba(X_test_grouped)[:, 1]
grouped_y = y_test_grouped.to_numpy()

roc_auc = roc_auc_score(grouped_y, grouped_test_scores)

def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return float(np.mean(y_true[order]))

p20 = precision_at_k(grouped_y, grouped_test_scores, 20)
p50 = precision_at_k(grouped_y, grouped_test_scores, 50)
p100 = precision_at_k(grouped_y, grouped_test_scores, 100)
base_rate = float(grouped_y.mean())

print("Validated client-grouped evidence")
print(f"Holdout rows: {len(grouped_y):,}")
print(f"Unseen clients: {len(set(test_grouped['client_id']))}")
print(f"Base decline rate: {base_rate:.3f} ({base_rate:.1%})")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Precision@20: {p20:.2f}")
print(f"Precision@50: {p50:.2f}")
print(f"Precision@100: {p100:.2f}")

Validated client-grouped evidence
Holdout rows: 6,163
Unseen clients: 7
Base decline rate: 0.511 (51.1%)
ROC-AUC: 0.6157
Precision@20: 0.70
Precision@50: 0.72
Precision@100: 0.70


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every item in the ranked queue requires human review before any content change is made.

The reviewer should:

1. Check the actual page and its recent performance context.
2. Confirm that the reason codes describe a meaningful review signal.
3. Check whether the page has important business, seasonal, or editorial context that the model does not capture.
4. Decide whether refresh, monitoring, or no action is appropriate.
5. Record the final human decision separately from the model score.

A high model score should increase review priority, but it should not determine the final action by itself.

### No-go list

The following actions should **not** be automated by this playbook:

- Automatically rewriting or deleting content.
- Automatically changing titles, headings, URLs, or metadata.
- Automatically changing internal links.
- Automatically publishing AI-generated content.
- Automatically changing content based only on a model score or reason code.
- Automatically claiming that a refresh will improve traffic, rankings, CTR, or engagement.
- Automatically treating the model score as a future-performance prediction.

These decisions require human review because the available validation supports ranking and prioritization, not causal or reliable future-performance claims.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Human-review and no-go checks

print("Human review requirement")
print("-------------------------")
print("All queue items require human review:", ranked_queue["human_review_required"].all())
print("Unique actions:", ranked_queue["action"].unique().tolist())

print("\nRequired queue fields:")
required_fields = [
    "rank",
    "content_id",
    "client_id",
    "model_score",
    "action",
    "reason_code",
    "confidence_note",
    "human_review_required"
]

for field in required_fields:
    print(f"{field}: {'OK' if field in ranked_queue.columns else 'MISSING'}")

print("\nNo-go automation rules:")
no_go_rules = [
    "No automatic content rewriting",
    "No automatic publishing",
    "No automatic deletion",
    "No automatic metadata/URL changes",
    "No automatic causal claims",
    "No automatic future-performance decisions"
]

for rule in no_go_rules:
    print("-", rule)

Human review requirement
-------------------------
All queue items require human review: True
Unique actions: ['review_for_refresh']

Required queue fields:
rank: OK
content_id: OK
client_id: OK
model_score: OK
action: OK
reason_code: OK
confidence_note: OK
human_review_required: OK

No-go automation rules:
- No automatic content rewriting
- No automatic publishing
- No automatic deletion
- No automatic metadata/URL changes
- No automatic causal claims
- No automatic future-performance decisions


### Cost/value thinking

The ranked queue is intended to help allocate limited human review time.

A practical workflow is to review the highest-ranked items first rather than reviewing every item equally. The value of the queue comes from concentrating attention on items with stronger ranking signals and clearer review reasons.

The model does not estimate the monetary value of a content refresh. Therefore, this playbook does not assign financial ROI or claim that a refresh will generate a specific amount of traffic or revenue.

Human reviewers should consider business importance, content quality, effort required, and evidence of a genuine content issue before approving a refresh.

A high model score alone is not sufficient to justify expensive work.

In [26]:
# Cost/value framework — qualitative only.
# No monetary ROI is inferred from the model.

cost_value_framework = pd.DataFrame([
    {
        "factor": "Model priority",
        "question": "Is this item highly ranked in the review queue?",
        "use": "Prioritize review order"
    },
    {
        "factor": "Evidence strength",
        "question": "Are there multiple meaningful review signals?",
        "use": "Assess whether deeper review is worthwhile"
    },
    {
        "factor": "Business importance",
        "question": "Is the content important to the site's goals?",
        "use": "Consider before allocating significant effort"
    },
    {
        "factor": "Refresh effort",
        "question": "How much human effort would a quality refresh require?",
        "use": "Compare effort with expected usefulness"
    },
    {
        "factor": "Content quality",
        "question": "Does human review identify a genuine content issue?",
        "use": "Decide whether action is justified"
    }
])

display(cost_value_framework)
print("Financial ROI is not estimated by this playbook.")

,factor,question,use
0,Model priority,Is this item highly ranked in the review queue?,Prioritize review order
1,Evidence strength,Are there multiple meaningful review signals?,Assess whether deeper review is worthwhile
2,Business importance,Is the content important to the site's goals?,Consider before allocating significant effort
3,Refresh effort,How much human effort would a quality refresh ...,Compare effort with expected usefulness
4,Content quality,Does human review identify a genuine content i...,Decide whether action is justified


Financial ROI is not estimated by this playbook.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring plan

This playbook is a research prototype, so monitoring is intended to detect when the ranking signal may no longer be reliable.

The following checks should be reviewed periodically:

| Check | Trigger | Response |
|---|---|---|
| Outcome rate | Observed decline rate changes substantially from the validation baseline of 51.1% | Review whether the target distribution has shifted |
| Ranking quality | Precision@20 or Precision@50 falls materially below the validated values of 0.70 and 0.72 | Investigate model performance before relying on the queue |
| Data distribution | Important input distributions change substantially from the development data | Check for data drift and changed content mix |
| Missingness | Important model inputs develop unexpected missing values | Investigate the data pipeline |
| Content mix | New content types or major shifts in existing content types appear | Review whether the model still represents the current data |
| Client mix | The queue is applied to substantially different client populations | Re-evaluate using client-grouped validation |

### Retrain triggers

Retraining should be considered when:

- ranking quality remains materially below the validated baseline;
- the underlying content population changes substantially;
- important features show persistent distribution drift;
- new content types become common;
- or the definition of the business problem or target label changes.

Retraining should not be triggered by a single unusual observation. A sustained change should be investigated first.

Any retrained model should be evaluated again using a client-grouped holdout before being used for a new action queue.

### Decay / refresh insight

The current model provides a directional ranking signal for the observed declining-label task. The data also includes freshness-related variables such as `content_age_days` and `days_since_last_update`, which can help identify content that may deserve review.

However, these variables do not establish that refreshing older content will improve performance. They should therefore be used as review signals rather than automatic refresh rules.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Monitoring baseline and retrain triggers

monitoring_baseline = {
    "decline_rate": base_rate,
    "roc_auc": roc_auc,
    "precision_at_20": p20,
    "precision_at_50": p50,
    "precision_at_100": p100,
}

print("Monitoring baseline")
print("-------------------")

for metric, value in monitoring_baseline.items():
    if "rate" in metric or "precision" in metric:
        print(f"{metric}: {value:.3f}")
    else:
        print(f"{metric}: {value:.4f}")

print("\nRetrain should be considered when:")
print("- Ranking quality remains materially below the validated baseline.")
print("- Important feature distributions show persistent drift.")
print("- The content population or content mix changes substantially.")
print("- New content types become common.")
print("- The target definition or business problem changes.")

print("\nRetrain verification:")
print("- Re-run client-grouped validation.")
print("- Compare ROC-AUC and Precision@K with the previous validated baseline.")
print("- Review false positives and false negatives.")
print("- Rebuild the action queue only after validation.")

Monitoring baseline
-------------------
decline_rate: 0.511
roc_auc: 0.6157
precision_at_20: 0.700
precision_at_50: 0.720
precision_at_100: 0.700

Retrain should be considered when:
- Ranking quality remains materially below the validated baseline.
- Important feature distributions show persistent drift.
- The content population or content mix changes substantially.
- New content types become common.
- The target definition or business problem changes.

Retrain verification:
- Re-run client-grouped validation.
- Compare ROC-AUC and Precision@K with the previous validated baseline.
- Review false positives and false negatives.
- Rebuild the action queue only after validation.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Paper-ready exports

The ranked queue generated in this notebook is exported for reuse in the research paper.

The queue contains the model ranking, proposed review action, reason codes, confidence note, and human-review requirement.

The exported queue is intended as a reproducible research artifact rather than a production decision system.

The queue is regenerated by the notebook and is not treated as a committed data file because the repository's CI checks prevent generated data files from being committed.

The paper should describe the queue as a human-review prioritization artifact and should not present it as an automated content optimization system.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 5 — Export the ranked queue for reuse in the paper
# Section 5 — Export and verify the ranked queues for the paper.

from pathlib import Path

outputs_dir = repo_path / "work" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

# Internal reproducible queue
queue_path = outputs_dir / "w07_ranked_action_queue.csv"
ranked_queue.to_csv(queue_path, index=False)

# Paper-safe queue without client identifiers
paper_queue = ranked_queue.drop(columns=["client_id"]).copy()
paper_queue_path = outputs_dir / "w07_ranked_action_queue_paper.csv"
paper_queue.to_csv(paper_queue_path, index=False)

print("Export verification")
print("-------------------")

print("Internal queue:")
print("Path:", queue_path)
print("Exists:", queue_path.exists())
print("Rows:", len(ranked_queue))
print("Columns:", len(ranked_queue.columns))

print("\nPaper-safe queue:")
print("Path:", paper_queue_path)
print("Exists:", paper_queue_path.exists())
print("Rows:", len(paper_queue))
print("Columns:", len(paper_queue.columns))
print("Contains client_id:", "client_id" in paper_queue.columns)

# Reload both files to verify they are valid CSV exports.
check_internal = pd.read_csv(queue_path)
check_paper = pd.read_csv(paper_queue_path)

assert len(check_internal) == 6163
assert len(check_paper) == 6163
assert check_internal["rank"].min() == 1
assert check_internal["rank"].max() == 6163
assert check_internal["human_review_required"].all()
assert "client_id" not in check_paper.columns

print("\nReload verification: PASSED")
print("Internal file size:",
      round(queue_path.stat().st_size / 1024, 2), "KB")
print("Paper-safe file size:",
      round(paper_queue_path.stat().st_size / 1024, 2), "KB")

Export verification
-------------------
Internal queue:
Path: /content/ML_internship_Qurban/work/outputs/w07_ranked_action_queue.csv
Exists: True
Rows: 6163
Columns: 8

Paper-safe queue:
Path: /content/ML_internship_Qurban/work/outputs/w07_ranked_action_queue_paper.csv
Exists: True
Rows: 6163
Columns: 7
Contains client_id: False

Reload verification: PASSED
Internal file size: 1164.08 KB
Paper-safe file size: 1055.74 KB


In [29]:
# Create a paper-safe version without client identifiers.

paper_queue = ranked_queue.drop(columns=["client_id"]).copy()

paper_queue_path = outputs_dir / "w07_ranked_action_queue_paper.csv"
paper_queue.to_csv(paper_queue_path, index=False)

print("Paper-safe export")
print("-----------------")
print("Path:", paper_queue_path)
print("Rows:", len(paper_queue))
print("Columns:", len(paper_queue.columns))
print("Contains client_id:", "client_id" in paper_queue.columns)

Paper-safe export
-----------------
Path: /content/ML_internship_Qurban/work/outputs/w07_ranked_action_queue_paper.csv
Rows: 6163
Columns: 7
Contains client_id: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.